# Regularizations

In [1]:
import numpy as np # 넘파이 임포트
import matplotlib.pyplot as plt # 시각화 라이브러리

import torch # 파이토치 임포트
import torch.nn as nn # 신경망 모듈
import torch.nn.functional as F # 신경망 함수
import torch.optim as optim # 최적화 알고리즘

In [2]:
from torchvision import datasets, transforms # 데이터셋 및 변환 도구

In [3]:
train = datasets.MNIST( # MNIST 학습 데이터
    '../data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(), # 텐서 변환
    ]),
)
test = datasets.MNIST( # MNIST 테스트 데이터
    '../data', train=False,
    transform=transforms.Compose([
        transforms.ToTensor(), # 텐서 변환
    ]),
)

In [4]:
x = train.data.float() / 255. # 데이터 정규화
y = train.targets # 레이블 설정

x = x.view(x.size(0), -1) # 차원 평탄화
print(x.shape, y.shape) # 크기 출력

input_size = x.size(-1) # 입력 크기
output_size = int(max(y)) + 1 # 출력 클래스 수

print('input_size: %d, output_size: %d' % (input_size, output_size)) # 크기 확인

In [5]:
# Train / Valid ratio
ratios = [.8, .2] # 데이터 분할 비율

train_cnt = int(x.size(0) * ratios[0]) # 학습 데이터 수
valid_cnt = int(x.size(0) * ratios[1]) # 검증 데이터 수
test_cnt = len(test.data) # 테스트 데이터 수
cnts = [train_cnt, valid_cnt] # 개수 리스트

print("Train %d / Valid %d / Test %d samples." % (train_cnt, valid_cnt, test_cnt))

indices = torch.randperm(x.size(0)) # 무작위 인덱스 생성

x = torch.index_select(x, dim=0, index=indices) # 데이터 섞기
y = torch.index_select(y, dim=0, index=indices) # 레이블 섞기

x = list(x.split(cnts, dim=0)) # 데이터 분할
y = list(y.split(cnts, dim=0)) # 레이블 분할

x += [(test.data.float() / 255.).view(test_cnt, -1)] # 테스트 데이터 추가
y += [test.targets] # 테스트 레이블 추가

for x_i, y_i in zip(x, y):
    print(x_i.size(), y_i.size()) # 최종 크기 확인

## Build Regularized Model & Optimizer

In [6]:
class Block(nn.Module): # 신경망 블록 클래스
    
    def __init__(self,
                 input_size,
                 output_size,
                 use_batch_norm=True,
                 dropout_p=.4):
        self.input_size = input_size # 입력 크기 저장
        self.output_size = output_size # 출력 크기 저장
        self.use_batch_norm = use_batch_norm # 배치 정규화 사용 여부
        self.dropout_p = dropout_p # 드롭아웃 확률
        
        super().__init__() # 부모 클래스 초기화
        
        def get_regularizer(use_batch_norm, size): # 정규화 도구 선택 함수
            return nn.BatchNorm1d(size) if use_batch_norm else nn.Dropout(dropout_p)
        
        self.block = nn.Sequential( # 시퀀셜 블록 구성
            nn.Linear(input_size, output_size), # 선형 계층
            nn.LeakyReLU(), # 활성화 함수
            get_regularizer(use_batch_norm, output_size), # 정규화 적용
        )
        
    def forward(self, x): # 순전파 과정
        y = self.block(x) # 블록 실행
        return y # 결과 반환

In [7]:
class MyModel(nn.Module): # 전체 모델 클래스
    
    def __init__(self,
                 input_size,
                 output_size,
                 use_batch_norm=True,
                 dropout_p=.4):
        
        super().__init__() # 초기화
        
        self.layers = nn.Sequential( # 다층 신경망 구성
            Block(input_size, 500, use_batch_norm, dropout_p), # 블록 1
            Block(500, 400, use_batch_norm, dropout_p), # 블록 2
            Block(400, 300, use_batch_norm, dropout_p), # 블록 3
            Block(300, 200, use_batch_norm, dropout_p), # 블록 4
            Block(200, 100, use_batch_norm, dropout_p), # 블록 5
            nn.Linear(100, output_size), # 최종 출력층
            nn.LogSoftmax(dim=-1), # 로그 소프트맥스
        )
        
    def forward(self, x): # 순전파 실행
        y = self.layers(x) # 모든 계층 통과
        return y # 예측값 반환

In [8]:
model = MyModel(input_size, # 모델 인스턴스 생성
                output_size,
                use_batch_norm=True)

model # 모델 구조 표시

In [9]:
crit = nn.NLLLoss() # 손실 함수 설정

In [10]:
optimizer = optim.Adam(model.parameters()) # 최적화 알고리즘 설정

## Move to GPU if it is available

In [ ]:
device = torch.device('cpu') # 기본 장치 설정
if torch.cuda.is_available(): # GPU 가능 시
    device = torch.device('cuda') # GPU로 변경

In [12]:
device # 사용 장치 확인

In [13]:
model = model.to(device) # 모델을 장치로 이동

x = [x_i.to(device) for x_i in x] # 데이터를 장치로 이동
y = [y_i.to(device) for y_i in y] # 레이블을 장치로 이동

## Train

In [14]:
n_epochs = 1000 # 총 에포크 수
batch_size = 256 # 배치 크기
print_interval = 10 # 출력 간격

In [ ]:
from copy import deepcopy # 깊은 복사 도구

lowest_loss = np.inf # 최소 손실 초기화
best_model = None # 최적 모델 초기화

early_stop = 50 # 조기 종료 조건
lowest_epoch = np.inf # 최소 손실 에포크

In [16]:
train_history, valid_history = [], [] # 히스토리 저장 리스트

for i in range(n_epochs): # 에포크 반복
    model.train() # 학습 모드 설정
    
    indices = torch.randperm(x[0].size(0)).to(device) # 무작위 인덱스
    x_ = torch.index_select(x[0], dim=0, index=indices) # 데이터 섞기
    y_ = torch.index_select(y[0], dim=0, index=indices) # 레이블 섞기
    
    x_ = x_.split(batch_size, dim=0) # 미니배치 분할
    y_ = y_.split(batch_size, dim=0) # 미니배치 레이블 분할
    
    train_loss, valid_loss = 0, 0 # 에포크 손실 초기화
    y_hat = []
    
    for x_i, y_i in zip(x_, y_): # 미니배치 반복
        y_hat_i = model(x_i) # 예측
        loss = crit(y_hat_i, y_i.squeeze()) # 손실 계산

        optimizer.zero_grad() # 기울기 초기화
        loss.backward() # 역전파
        optimizer.step() # 가중치 업데이트
        
        train_loss += float(loss) # 손실 누적

    train_loss = train_loss / len(x_) # 평균 학습 손실
        
    model.eval() # 평가 모드 설정
    with torch.no_grad(): # 기울기 계산 비활성화
        x_ = x[1].split(batch_size, dim=0) # 검증 데이터 분할
        y_ = y[1].split(batch_size, dim=0) # 검증 레이블 분할
        
        valid_loss = 0
        for x_i, y_i in zip(x_, y_): # 검증 반복
            y_hat_i = model(x_i) # 예측
            loss = crit(y_hat_i, y_i.squeeze()) # 손실 계산
            valid_loss += float(loss) # 손실 누적
            y_hat += [y_hat_i] # 결과 저장
            
    valid_loss = valid_loss / len(x_) # 평균 검증 손실
    train_history += [train_loss] # 기록 추가
    valid_history += [valid_loss] # 기록 추가
        
    if (i + 1) % print_interval == 0: # 결과 출력
        print('Epoch %d: train loss=%.4e  valid_loss=%.4e  lowest_loss=%.4e' % (
            i + 1, train_loss, valid_loss, lowest_loss,
        ))
        
    if valid_loss <= lowest_loss: # 최소 손실 갱신
        lowest_loss = valid_loss
        lowest_epoch = i
        best_model = deepcopy(model.state_dict()) # 최적 가중치 저장
    else:
        if early_stop > 0 and lowest_epoch + early_stop < i + 1: # 조기 종료 확인
            print("There is no improvement during last %d epochs." % early_stop)
            break

print("The best validation loss from epoch %d: %.4e" % (lowest_epoch + 1, lowest_loss))
model.load_state_dict(best_model) # 최적 가중치 복원

## Loss History

In [17]:
plot_from = 0

plt.figure(figsize=(20, 10)) # 그래프 크기 설정
plt.grid(True) # 그리드 표시
plt.title("Train / Valid Loss History") # 제목 설정
plt.plot( # 그래프 그리기
    range(plot_from, len(train_history)), train_history[plot_from:],
    range(plot_from, len(valid_history)), valid_history[plot_from:],
)
plt.yscale('log') # Y축 로그 스케일
plt.show() # 출력

## Let's see the result!

In [18]:
test_loss = 0
y_hat = []

model.eval() # 평가 모드
with torch.no_grad(): # 기울기 무시
    x_ = x[-1].split(batch_size, dim=0) # 테스트 데이터 분할
    y_ = y[-1].split(batch_size, dim=0) # 테스트 레이블 분할

    for x_i, y_i in zip(x_, y_): # 테스트 반복
        y_hat_i = model(x_i) # 예측
        loss = crit(y_hat_i, y_i.squeeze()) # 손실 계산
        test_loss += loss # 손실 합산
        y_hat += [y_hat_i] # 결과 저장

test_loss = test_loss / len(x_) # 평균 테스트 손실
y_hat = torch.cat(y_hat, dim=0) # 결과 통합

print("Test loss: %.4e" % test_loss) # 결과 출력

In [19]:
correct_cnt = (y[-1].squeeze() == torch.argmax(y_hat, dim=-1)).sum() # 정답 수 계산
total_cnt = float(y[-1].size(0)) # 전체 수 계산

print('Accuracy: %.4f' % (correct_cnt / total_cnt)) # 정확도 출력

In [20]:
import pandas as pd # 판다스 임포트
from sklearn.metrics import confusion_matrix # 혼동 행렬 도구

In [21]:
pd.DataFrame( # 혼동 행렬 출력
    confusion_matrix(
        y[-1].cpu().numpy(),  # 실젯값
        torch.argmax(y_hat, dim=-1).cpu().numpy()  # 예측값
    ),
    index=['true_%d' % i for i in range(10)],
    columns=['pred_%d' % i for i in range(10)]
)